In [6]:
# Zenodo → Google Drive SMART TRANSFER (resumable + status check)
from google.colab import drive
import requests, urllib3, os, time

# --- SETTINGS ---
zenodo_id = "14979685"
target_dir = "/content/drive/MyDrive/SLAM_Lite/zenodo_14979685"
threads = 8                 # parallel connections
segment_size = "10M"        # chunk size per connection
# ----------------

# 1. Mount Drive
drive.mount('/content/drive')

# 2. Install aria2
!apt -qq install -y aria2

# 3. Create destination folder
os.makedirs(target_dir, exist_ok=True)

# 4. Fetch Zenodo file list
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
api_url = f"https://zenodo.org/api/records/{zenodo_id}"
print(f"Fetching metadata from {api_url} ...")
r = requests.get(api_url, verify=False, timeout=30)
r.raise_for_status()
data = r.json()

# 5. Build link list and check what is already downloaded
link_file = "/content/zenodo_links.txt"
existing_files = os.listdir(target_dir)
to_download = []

with open(link_file, "w") as f:
    for file in data.get("files", []):
        name = file["key"]
        url = file["links"]["self"] + "?download=1"
        if name in existing_files:
            size = os.path.getsize(os.path.join(target_dir, name)) / 1e6
            print(f"Already present: {name} ({size:.2f} MB)")
        else:
            to_download.append(url)
            f.write(url + "\n")

print(f"\n{len(existing_files)} files already present in Drive.")
print(f"{len(to_download)} files remaining to download.\n")

# 6. Start resumable download
if len(to_download) > 0:
    print("Starting resumable download in 5 seconds...")
    time.sleep(5)
    start = time.time()
    !aria2c -i /content/zenodo_links.txt -d "$target_dir" \
        -x {threads} -s {threads} -k {segment_size} --continue=true
    end = time.time()
    print(f"\nTransfer complete (elapsed {((end-start)/60):.1f} minutes)")
else:
    print("All files already downloaded.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
aria2 is already the newest version (1.36.0-1).
0 upgraded, 0 newly installed, 0 to remove and 41 not upgraded.
Fetching metadata from https://zenodo.org/api/records/14979685 ...
Already present: README.md (0.00 MB)
Already present: DJI_0034.MOV (1035.55 MB)
Already present: DJI_0028.MOV (846.07 MB)
Already present: DJI_0019.MOV (896.62 MB)
Already present: DJI_0018.MOV (698.38 MB)
Already present: DJI_0048.MOV (1147.74 MB)
Already present: DJI_0047.MOV (1100.90 MB)
Already present: DJI_0033.MOV (1136.43 MB)
Already present: DJI_0046.MOV (1475.13 MB)
Already present: DJI_0044.MOV (1100.23 MB)
Already present: DJI_0056.MOV (5791.42 MB)
Already present: DJI_0049.MOV (1221.62 MB)
Already present: DJI_0051.MOV (1285.03 MB)
Already present: DJI_0037.MOV (1074.45 MB)
Already present: DJI_0016.MOV (676.53 MB)
Already present: DJI_0027.MOV (952.20 MB)
Already present